# 05 — Vector Store & RAG Pipeline Evaluation

**Phase 5 deliverable.**

Evaluates the hybrid retrieval pipeline (dense + sparse + RRF + reranker) against a
labeled test set. Runs the ablation study comparing four configurations:

| Configuration | Dense | Sparse | RRF | Reranker |
|---|---|---|---|---|
| Dense only | ✓ | | | |
| Sparse only | | ✓ | | |
| Hybrid (RRF) | ✓ | ✓ | ✓ | |
| Hybrid + reranker | ✓ | ✓ | ✓ | ✓ |

**Targets:** Recall@5 > 0.70, MRR > 0.65

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

from src import config
from src.rag.indexer import load_chroma_collection, load_bm25_index
from src.nlp.embedder import embed_query
from src.rag.retriever import retrieve_dense, retrieve_sparse, reciprocal_rank_fusion
from src.rag.reranker import rerank

MODEL_DIR = ROOT / "models" / "sentence_transformer"

## 1. Load Indexes

In [ ]:
collection = load_chroma_collection(config.CHROMADB_DIR, config.CHROMA_COLLECTION_NAME)
print(f"ChromaDB: {collection.count():,} documents")

bm25_index, bm25_chunks = load_bm25_index(config.VECTORSTORE_DIR / "bm25_index.pkl")
print(f"BM25: {len(bm25_chunks):,} chunks")

## 2. Load Retrieval Test Set

In [ ]:
with open(config.EVALUATION_DIR / "retrieval_test_set.json") as f:
    test_set = json.load(f)

queries = test_set["queries"]
print(f"Loaded {len(queries)} evaluation queries")
print(f"Query types: {set(q['query_type'] for q in queries)}")
print("\nSample:")
print(json.dumps(queries[0], indent=2, ensure_ascii=False))

## 3. Retrieval Metrics — Helper Functions

In [ ]:
def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """Proportion of relevant docs found in the top-k retrieved."""
    if not relevant_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant_ids) / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list[str], relevant_ids: set[str]) -> float:
    """Reciprocal of the rank of the first relevant document."""
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """Normalised Discounted Cumulative Gain at k."""
    import math
    dcg = sum(
        1.0 / math.log2(rank + 2)
        for rank, doc_id in enumerate(retrieved_ids[:k])
        if doc_id in relevant_ids
    )
    ideal = sum(1.0 / math.log2(rank + 2) for rank in range(min(len(relevant_ids), k)))
    return dcg / ideal if ideal > 0 else 0.0


def evaluate_config(
    queries: list[dict],
    retrieve_fn,  # callable(query_str) -> list[RetrievedChunk]
    ks: tuple[int, ...] = (5, 10),
) -> dict:
    """Run retrieval for every query and aggregate Recall@K, MRR, NDCG@10."""
    recall_k = {k: [] for k in ks}
    mrr_scores = []
    ndcg_scores = []

    for q in queries:
        relevant = set(q["relevant_chunk_ids"])
        results = retrieve_fn(q["query"])
        ids = [c.chunk_id for c in results]

        for k in ks:
            recall_k[k].append(recall_at_k(ids, relevant, k))
        mrr_scores.append(reciprocal_rank(ids, relevant))
        ndcg_scores.append(ndcg_at_k(ids, relevant, 10))

    import numpy as np
    return {
        **{f"Recall@{k}": float(np.mean(recall_k[k])) for k in ks},
        "MRR": float(np.mean(mrr_scores)),
        "NDCG@10": float(np.mean(ndcg_scores)),
    }

## 4. Ablation Study

In [ ]:
import numpy as np
import pandas as pd

# --- Configuration 1: Dense only ---
def dense_only(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    return retrieve_dense(emb, collection, top_k=10)

# --- Configuration 2: Sparse only ---
def sparse_only(query: str):
    return retrieve_sparse(query, bm25_index, bm25_chunks, top_k=10)

# --- Configuration 3: Hybrid RRF ---
def hybrid_rrf(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    dense = retrieve_dense(emb, collection, top_k=20)
    sparse = retrieve_sparse(query, bm25_index, bm25_chunks, top_k=20)
    return reciprocal_rank_fusion(dense, sparse, k=60, top_n=10)

# --- Configuration 4: Hybrid + Reranker ---
def hybrid_reranker(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    dense = retrieve_dense(emb, collection, top_k=20)
    sparse = retrieve_sparse(query, bm25_index, bm25_chunks, top_k=20)
    fused = reciprocal_rank_fusion(dense, sparse, k=60, top_n=10)
    return rerank(query, fused, top_k=5)

configs = [
    ("Dense only (fine-tuned BERTimbau)", dense_only),
    ("Sparse only (BM25)",               sparse_only),
    ("Hybrid (RRF)",                     hybrid_rrf),
    ("Hybrid + Reranker",                hybrid_reranker),
]

results = {}
for name, fn in configs:
    print(f"Evaluating: {name} ...")
    results[name] = evaluate_config(queries, fn)
    print(f"  {results[name]}")

df = pd.DataFrame(results).T
df = df[["Recall@5", "Recall@10", "MRR", "NDCG@10"]]
df = df.round(4)
print("\n=== Ablation Results ===")
print(df.to_string())

## 5. Results Table

In [ ]:
# Styled table for reporting
TARGET_RECALL5 = 0.70
TARGET_MRR = 0.65

print(df.to_markdown())

best = df.loc["Hybrid + Reranker"]
print(f"\nRecall@5 target (>0.70): {best['Recall@5']:.3f} — {'PASS' if best['Recall@5'] >= TARGET_RECALL5 else 'FAIL'}")
print(f"MRR target    (>0.65): {best['MRR']:.3f} — {'PASS' if best['MRR'] >= TARGET_MRR else 'FAIL'}")

## 6. End-to-End RAG — Sample Answers

In [ ]:
from src.rag.generator import generate_answer

sample_queries = [
    "Qual foi a estratégia de crescimento da Petrobras em 2023?",
    "Como a Vale descreveu os riscos ambientais em seus relatórios?",
    "Qual foi o impacto da taxa de juros no desempenho do Itaú Unibanco?",
]

for query in sample_queries:
    print(f"\nQ: {query}")
    print("-" * 60)
    chunks = hybrid_reranker(query)
    answer = generate_answer(query, chunks)
    print(answer)
    print()